In [1]:
import sqlite3            # open the database
import pandas as pd       # read SQL into a DataFrame
import numpy as np        # for the mean/std etc.
import matplotlib.pyplot as plt   # or seaborn, plotly, etc.

In [ ]:
# Get connection to the database
conn = sqlite3.connect('spectra.db')
df   = pd.read_sql_query("SELECT SNR FROM spectra", conn)
conn.close()

#Compute stats
mean = df['SNR'].mean()
std = df['SNR'].std()

#Plotting histogram
plt.hist(df['SNR'], bins=1000, alpha=0.7, color='blue')
plt.axvline(mean-3*std, color='r', linestyle='--',
            label='mean - 3σ')
plt.legend("Distribution of SNR values with mean - 3σ threshold")
plt.xlabel('SNR')
plt.ylabel('count')
plt.show()

In [ ]:
# compute cutoff and filter the dataframe
cutoff = mean - 3*std
print(f"cutoff value: {cutoff}")
# rows to keep
cleaned = df[df['SNR'] > cutoff]

# open connection again to remove the others from the SQL table
del_conn = sqlite3.connect('spectra.db')
del_cur = del_conn.cursor()

# delete rows with SNR <= cutoff
# note: SQLite will treat NULL as not <=, so they won't be removed
sql_delete = "DELETE FROM spectra WHERE SNR <= ?"
del_cur.execute(sql_delete, (cutoff,))
del_conn.commit()

del_conn.close()

print(f"original rows: {len(df)}, remaining after deletion: {len(cleaned)}")
# if you want the cleaned dataframe written back entirely instead of deleting, you could do:
# cleaned.to_sql('spectra', sqlite3.connect('spectra.db'), if_exists='replace', index=False)
